# Lenormand B16 — Factor Hidden-State Readout Gate

目标不是再训练27B，而是验证：Qwen3.8 Factor verifier 的第48/64层是否已经编码了正确判断，只是最后的`A/B`语言模型头读错。

- 冻结已有 Full64 Fold-0 adapter；
- 对所有`帖子×24标签`提示读取第48/64层和原A/B margin；
- 只在 outer-train 用户上训练共享、label-aware线性readout；
- inner GroupKFold产生无泄漏训练分数与阈值；
- 只在Fold 0做screening。未达到 `Macro-F1 +0.008`、`Macro-AP +0.004` 自动停止。

A100 80GB预计3–5小时。每128个prompt保存到Drive，断线后重跑主格可恢复。


In [ ]:
#@title 0A. 新runtime安装依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 kernels（完成后重启session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('Runtime → Restart session；重启后从第1格开始。')


In [ ]:
#@title 1. Drive、冻结产物与模块
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, shutil, subprocess, sys, time

ROOT=Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH=ROOT/'train.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH=ROOT/'ieee/train.xlsx'
Q14_OOF=ROOT/'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
Q38_OOF=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/Q38_FULL64_OOF.npz'
FULL64_CONFIG=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/FULL64_CONFIG.json'
FACTOR_ROOT=ROOT/'results/B4_Q38F_FULL64_KERNEL_FOLD0/FULL64_FACTOR_FOLD0'
ADAPTER=FACTOR_ROOT/'fold_0/verifier/adapter_final'
OUT=ROOT/'results/B16_FACTOR_LATENT_GATE/fold_0'
OUT.mkdir(parents=True,exist_ok=True)

MODULE_MARKERS={
 'b1_experiments.py':None,
 'b4p_anchor_verifier.py':'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
 'b15_latent_readout.py':'B15_RUNTIME_REVISION = "2026-08-30.latent-risk-readout-v1"',
 'b16_factor_latent_readout.py':'B16_RUNTIME_REVISION = "2026-08-30.factor-hidden-readout-gate-v1"',
}
stale=[]
for name,marker in MODULE_MARKERS.items():
    path=ROOT/name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖：',stale)
    uploaded=files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/'+name,ROOT/name)

required=[TRAIN_PATH,Q14_OOF,Q38_OOF,FULL64_CONFIG,ADAPTER/'adapter_config.json']
missing=[str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('缺少冻结产物：\n'+'\n'.join(missing))
sys.path.insert(0,str(ROOT))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
print({'adapter':str(ADAPTER),'output':str(OUT),'free_gb':round(shutil.disk_usage(ROOT).free/2**30,2)})


In [ ]:
#@title 2. 环境、数据、Full64配置和基线对齐
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b15_latent_readout as b15
import b16_factor_latent_readout as b16
importlib.reload(b1);importlib.reload(b4);importlib.reload(b15);importlib.reload(b16)

gpu_gb=torch.cuda.get_device_properties(0).total_memory/2**30
kernel=b4.qwen35_kernel_status()
print({'transformers':transformers.__version__,'torch':torch.__version__,
       'sklearn':sklearn.__version__,'gpu_gb':gpu_gb,'kernel':kernel})
assert gpu_gb>=70
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], '运行0B并重启session'
torch.set_float32_matmul_precision('high')

bundle=b1.load_training_data(ROOT,TRAIN_PATH)
anchor=b16.load_factor_anchor(bundle,Q14_OOF,Q38_OOF,q38_weight=0.75)
folds=anchor['folds']
verifier_cfg=b16.load_full64_config(FULL64_CONFIG)
assert verifier_cfg.verifier_model=='Qwen/Qwen3.8-27B'
assert verifier_cfg.lora_last_n_layers is None
print({'rows':len(bundle.texts),'fold_sizes':np.bincount(folds).tolist(),
       'model':verifier_cfg.verifier_model,'max_length':verifier_cfg.max_length,
       'attention':verifier_cfg.attention_implementation})


In [ ]:
#@title 3. 语义检索缓存（通常直接resume）
corpus=b4.training_corpus(bundle)
cache_candidates=[
 ROOT/'results/B4_Q38F_FULL64_KERNEL_FOLD0/semantic_cache/train',
 ROOT/'results/B4_Q38F_FAST3/semantic_cache/train',
]
cache_path=next((path for path in cache_candidates if path.exists()),cache_candidates[0])
semantic_cache=b4.prepare_semantic_cache(corpus,verifier_cfg,cache_path)
print('Semantic cache:',cache_path)


In [ ]:
#@title 4. 冻结B16配置与耗时说明
CFG=b16.FactorLatentConfig(
    fold=0,
    selected_layers=(47,63),
    c_grid=(0.001,0.01,0.1),
    blend_alphas=(0.25,0.50,0.75,1.00),
    inner_splits=3,
    extraction_batch_size=2,
    extraction_chunk_size=128,
    gate_macro_f1=0.008,
    gate_macro_ap=0.004,
    gate_tail_floor=-0.005,
    seed=42,
)
b16.json_dump(dataclasses.asdict(CFG),OUT/'B16_CONFIG.json')
print(dataclasses.asdict(CFG))
print('主格约3–5小时。可断线；重连后重跑1–5格，已保存的128-prompt chunks不会重算。')


In [ ]:
#@title 5. 主实验：抽hidden → inner crossfit probe → Fold-0 gate
started=time.perf_counter()
decision=b16.run_fold_gate(
    bundle=bundle,
    anchor=anchor,
    semantic_cache=semantic_cache,
    verifier_config=verifier_cfg,
    adapter_path=ADAPTER,
    output_dir=OUT,
    config=CFG,
)
print('\n=== SUMMARY ===')
display(pd.read_csv(OUT/'B16_FOLD_SUMMARY.csv'))
print('\n=== DECISION ===')
print(json.dumps(decision,indent=2,default=str))
print({'elapsed_hours':round((time.perf_counter()-started)/3600,2)})
if decision['passed']:
    print('PASS：把报告发回；下一步冻结 layer/C/alpha，只跑Fold 1/2 confirmation。')
else:
    print('FAIL：停止B16，不跑另外两折。')


In [ ]:
#@title 6. 打包轻量报告（不含hidden cache）
package=Path('/content/B16_FACTOR_LATENT_GATE_REPORT')
if package.exists(): shutil.rmtree(package)
package.mkdir(parents=True)
for name in ('B16_CONFIG.json','B16_PROBE_SELECTION.csv','B16_FOLD_SUMMARY.csv',
             'B16_PER_LABEL.csv','B16_FOLD_DECISION.json','B16_FOLD_OUTPUTS.npz'):
    path=OUT/name
    if path.exists(): shutil.copy2(path,package/name)
archive=shutil.make_archive('/content/B16_FACTOR_LATENT_GATE_REPORT','zip',package)
print(archive)
files.download(archive)
